# Options Volatility Modeling
## Comparing Black-Scholes and SABR Models on Real Market Data

## 1. Introduction
In this notebook we explore the limitations of the Black-Scholes model 
and demonstrate how the SABR stochastic volatility model better captures 
real market behavior. We use real SPY options data to extract implied 
volatilities, visualize the volatility smile, and compare model fits 
quantitatively using RMSE.

## 2. Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import brentq, minimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore') #suppress all warning messages

print("All libraries imported successfully")

All libraries imported successfully


## 3. Fetching SPY Options Data
We fetch real options data for SPY (S&P 500 ETF) using yfinance. 
SPY is ideal because it is heavily traded, meaning liquid options 
with reliable prices.

Look at all available methods and attributes  
`dir(yf)`  
Full documentaion of a particular method  
`help(yf.Ticker)`  
Available info for a specific ticker object  
`spy = yf.Ticker("SPY")`   
`dir(spy)`

In [2]:
# fetch the SPY options data
spy = yf.Ticker("SPY")

spy.info          # general info about the stock
spy.history()     # historical price data
spy.options       # available expiry dates
spy.option_chain  # options data for a specific expiry

<bound method Ticker.option_chain of yfinance.Ticker object <SPY>>

In [3]:
# list of all available methods and attributes
#dir(spy)

# full documentation
#help(spy.history)

In [4]:
# Get the current stock price
current_price = spy.history(period = "1d")['Close'].iloc[-1]
print(f'Current SPY price: {current_price:.2f}')

# See available expiry dates
expiry_dates = spy.options
print(f'\nAvailable expiry dates:')
for i, date in enumerate(expiry_dates[:10]): #show first 10
    print(f'{i+1}:{date}')

Current SPY price: 686.38

Available expiry dates:
1:2026-03-03
2:2026-03-04
3:2026-03-05
4:2026-03-06
5:2026-03-09
6:2026-03-10
7:2026-03-11
8:2026-03-12
9:2026-03-13
10:2026-03-20


In [5]:
#get all expiry dates
for i, date in enumerate(expiry_dates):
    print(f'{i+1}:{date}')

1:2026-03-03
2:2026-03-04
3:2026-03-05
4:2026-03-06
5:2026-03-09
6:2026-03-10
7:2026-03-11
8:2026-03-12
9:2026-03-13
10:2026-03-20
11:2026-03-27
12:2026-03-31
13:2026-04-02
14:2026-04-10
15:2026-04-17
16:2026-04-30
17:2026-05-15
18:2026-05-29
19:2026-06-18
20:2026-06-30
21:2026-07-31
22:2026-08-21
23:2026-08-31
24:2026-09-18
25:2026-09-30
26:2026-12-18
27:2026-12-31
28:2027-01-15
29:2027-03-19
30:2027-06-17
31:2027-12-17
32:2028-01-21
33:2028-06-16
34:2028-12-15


### Why We Choose a 30-60 Day Expiry Date

When selecting an options expiry date for volatility analysis, the 
time to expiry (also called DTE - Days to Expiry) matters significantly.

**Too short (< 30 days):**
- Options near expiry are dominated by gamma risk
- Implied volatilities become unstable and noisy
- Prices change very rapidly and erratically as expiry approaches,
  making smile fitting unreliable

**Too long (> 60 days):**
- Lower trading volume and wider bid-ask spreads i.e. fewer people
  are trading them
- Prices are less reliable due to lower liquidity
- Volatility smile becomes flatter and less interesting to analyze

**Sweet spot (30-60 days):**
- Enough time value to produce a meaningful volatility smile
- High liquidity means reliable, clean prices
- Stable implied volatilities suitable for model calibration

We choose **April 17, 2026** (46 days out from today, March 02, 2026) because:
- Falls in the middle of our 30-60 day target range
- It is the **third Friday of April** - a standard monthly expiry date
- Monthly expiries consistently attract the highest trading volume
  because institutional hedging strategies are structured around 
  monthly cycles, meaning tighter bid-ask spreads and more 
  reliable prices